# 14. Event Streaming — Observe Deep Agents as structured events

Streaming is not only a UI feature. For agent systems, structured events are the audit trail that explains what happened during planning, tool use, generation, and completion.

**Learning goals**
- Model agent progress as typed events.
- Consume events incrementally and update UI state.
- Use streaming as an operational debugging surface.


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

In [ ]:
# LangSmith / Langfuse setup — disabled when keys are not present.
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGSMITH_PROJECT", "agent-notebooks")

langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}

## 14.1 Event model

A useful event stream has stable event types and predictable payloads. This makes it possible to build logs, UIs, and monitors on top of the same execution trace.


In [ ]:
events = [
    {"type": "todo", "payload": {"task": "outline", "status": "done"}},
    {"type": "tool", "payload": {"name": "read_file", "status": "done"}},
    {"type": "subagent", "payload": {"name": "researcher", "status": "running"}},
    {"type": "message", "payload": {"text": "Drafting the first version."}},
]

len(events)

## 14.2 Build an event consumer

The consumer turns raw events into readable progress. Keep this logic small and deterministic so it remains trustworthy during incidents.


In [ ]:
def project_event(event: dict) -> str:
    kind = event["type"]
    payload = event["payload"]
    if kind == "tool":
        return f"tool:{payload['name']}:{payload['status']}"
    if kind == "subagent":
        return f"subagent:{payload['name']}:{payload['status']}"
    return f"{kind}:{payload}"

[project_event(event) for event in events]

## 14.3 Accumulate UI state

Most interfaces need the latest message, tool status, and completion state rather than a raw event dump. Accumulating UI state makes the stream usable for learners and operators.


In [ ]:
state = {"todos": [], "tools": [], "subagents": [], "messages": []}
for event in events:
    bucket = event["type"] + "s"
    if bucket in state:
        state[bucket].append(event["payload"])

state

## 14.4 Operational checklist

Streaming should be designed with failure modes in mind. Check event ordering, error events, redaction, and replayability before relying on it in production.


---

## Summary

| Item | Content |
|---|---|
| **Covered** | event logs, UI projections, subagent/tool/todo events, and replayable state |
| **Core idea** | Start from a small deterministic contract before adding model calls or external services. |
| **Next step** | Follow the linked course notebooks and official reference notes listed in this chapter. |

## Reference docs

- [`event-streaming.md`](../../docs/deepagents/event-streaming.md)
- [`streaming.md`](../../docs/deepagents/streaming.md)
- [`event-streaming.md`](../../docs/langchain/event-streaming.md)
